# ETL Silver para Gold

Transforma a BigTable Silver validada em tabelas dimensionais e marts analiticos da camada Gold.

Modelo gerado:

- `dim_tempo`
- `dim_regiao_administrativa`
- `fato_mapa_cidadania`
- `mart_indicadores_territoriais`
- `mart_acesso_informacao`


## 1. Setup

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "Transformer":
    ROOT_DIR = ROOT_DIR.parent

sys.path.insert(0, str(ROOT_DIR))

from src.transformations.gold import construir_gold, salvar_gold

SILVER_PATH = ROOT_DIR / "Data Layer" / "silver" / "tb_mapa_cidadania_ra_ano_silver.csv"
GOLD_DIR = ROOT_DIR / "Data Layer" / "gold"

SILVER_PATH, GOLD_DIR

## 2. Extract

In [ ]:
df_silver = pd.read_csv(SILVER_PATH, encoding="utf-8-sig")

print(f"Silver: {len(df_silver)} linhas, {len(df_silver.columns)} colunas")
print(f"RAs: {df_silver['regiao_administrativa'].nunique()}")
print(f"Anos: {sorted(df_silver['ano'].unique())}")

## 3. Transform

In [ ]:
gold = construir_gold(df_silver)

for nome_tabela, df in gold.items():
    print(f"{nome_tabela}: {len(df)} linhas, {len(df.columns)} colunas")

## 4. Validate

In [ ]:
assert len(gold["dim_tempo"]) == 3
assert len(gold["dim_regiao_administrativa"]) == 33
assert len(gold["fato_mapa_cidadania"]) == 99
assert len(gold["mart_indicadores_territoriais"]) == 33
assert len(gold["mart_acesso_informacao"]) == 3

fato = gold["fato_mapa_cidadania"]
assert fato[["sk_regiao_administrativa", "sk_tempo"]].notna().all().all()
assert fato.duplicated(["sk_regiao_administrativa", "sk_tempo"]).sum() == 0

ras = set(gold["dim_regiao_administrativa"]["regiao_administrativa"])
assert "Sudoeste_Octogonal" not in ras
assert "Sol Nascente_Pôr do Sol" not in ras
assert "Sol Nascente/Pôr do Sol" in ras

print("Validacoes Gold concluidas")

## 5. Load

In [ ]:
salvar_gold(gold, GOLD_DIR)

for caminho in sorted(GOLD_DIR.glob("*.csv")):
    if caminho.name != "tb_mapa_cidadania_ra_ano_silver.csv":
        print(caminho.relative_to(ROOT_DIR))